# 02 — Robust regression for the hinge model (fixing `s_t` / `s_{t-1}` collinearity)

The hinge model fits well, but the OLS coefficients aren't trustworthy because **`s_t` and `s_{t-1}` are almost collinear**. Consecutive seller quotes are close to each other by construction — the seller rarely jumps far between turns. When we try to disentangle the contribution of `s_{t-1}` from the contribution of the hinges, OLS allocates arbitrarily, and the coefficient on `s_{t-1}` jumps around across bootstrap samples.

This notebook does three things:

1. **Diagnose** the collinearity — correlation matrix, condition number, VIF, OLS coefficient bootstrap.
2. **Reparameterize** — predict `Δs = s_t − s_{t-1}` instead of `s_t`. This removes the trivial autoregressive part exactly. Under the assumed functional form, the hinge coefficients are identical in the two parameterizations.
3. **Regularize** — compare OLS with Ridge, PLS (Partial Least Squares), and PCR (Principal-Component Regression) in the Δ-parameterization. These are designed for collinear predictors. We show that while the point estimates of individual coefficients can differ, the *predictions* and the *identified functional form* agree.

**Notation recap.** `m_t, s_{t-1}, s_t` are merchant offer, seller's previous quote, and seller's next quote. `v = cost + internal_cost` is the break-even. Hinges are
$$(m_t - v)_+ = \max(m_t - v, 0), \qquad (v - m_t)_+ = \max(v - m_t, 0).$$
We write `diff_pos = (m_t − v)_+`, `diff_neg = (v − m_t)_+`, `ds_pos = max(s_{t-1} − s_{t-2}, 0)`, `ds_neg = max(s_{t-2} − s_{t-1}, 0)`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, RidgeCV, ElasticNetCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor

from analysis_common import (
    load_all_runs, add_hinge_features, NEGOTIATION_MASK, setup_plot_defaults,
)
setup_plot_defaults()

df = load_all_runs('results_0416b')
df = add_hinge_features(df)

# Analysis subset: Negotiation region, early rounds, needs lag-2 to exist.
sub = df[NEGOTIATION_MASK(df) & (df['round_t'] <= 8)].copy()
sub = sub.dropna(subset=['s_prev2', 'ds_prev'])
print(f'Analysis subset: {len(sub):,} rows / {sub["run_idx"].nunique()} runs')

## 1. Diagnosing the collinearity

We build the full feature set used by the successful models in your example notebooks and inspect its conditioning.

**Feature set (full hinge model).**
$$\text{features} = \{ \text{diff\_pos},\, \text{diff\_neg},\, s_{t-1},\, m_{t-1},\, \text{ds\_pos},\, \text{ds\_neg} \}$$

In [ ]:
FEATURES = ['diff_pos', 'diff_neg', 's_prev', 'm_prev', 'ds_pos', 'ds_neg']
X = sub[FEATURES].values
y = sub['s_t'].values
groups = sub['run_idx'].values
print('Feature columns :', FEATURES)
print('y (target)      : s_t')

corr = pd.DataFrame(X, columns=FEATURES).corr()
plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1)
plt.title('Feature correlation matrix')
plt.tight_layout(); plt.show()

# Correlation of each feature with the target
corr_y = pd.DataFrame({'feature': FEATURES,
                       'corr_with_s_t': [np.corrcoef(X[:, j], y)[0, 1]
                                          for j in range(X.shape[1])]})
display(corr_y.round(3))

### Condition number and VIF

**Condition number** κ of the standardized design matrix measures how much numerical trouble OLS is in. Rules of thumb: κ ≤ 30 is fine, κ > 100 is already serious multicollinearity.

**VIF_j** for feature `j` equals `1 / (1 − R²_j)` where `R²_j` is the R² of regressing feature `j` on all other features. VIF > 5 is suspicious; VIF > 10 is a red flag.

In [ ]:
Xs = StandardScaler().fit_transform(X)
sing = np.linalg.svd(Xs, compute_uv=False)
kappa = sing[0] / sing[-1]
print(f'Singular values of standardized X: {sing.round(3)}')
print(f'Condition number κ(X) = {kappa:.1f}')

vifs = pd.DataFrame({
    'feature': FEATURES,
    'VIF': [variance_inflation_factor(Xs, j) for j in range(Xs.shape[1])],
}).round(2)
display(vifs)
print('\nNote: if s_t is in the target, s_prev dominates because y ≈ s_prev + small.\n'
      'We will correct this by switching to Δs = s_t − s_prev as the target.')

### OLS coefficient instability — bootstrap

The clearest symptom of collinearity is that small changes in the data lead to big swings in individual coefficients while predictions stay nearly the same. We bootstrap **at the run level** (grouped bootstrap) so entire runs enter or leave together — the only honest version when rows within a run are correlated.

In [ ]:
rng = np.random.default_rng(0)
unique_runs = np.unique(groups)
n_boot = 200
coefs = np.zeros((n_boot, len(FEATURES)))
intercepts = np.zeros(n_boot)
for b in range(n_boot):
    sampled_runs = rng.choice(unique_runs, size=len(unique_runs), replace=True)
    mask = np.isin(groups, sampled_runs)
    # np.isin deduplicates multiplicities; draw each run with replacement
    # properly by stacking its rows
    idxs = np.concatenate([np.where(groups == r)[0] for r in sampled_runs])
    ols = LinearRegression().fit(X[idxs], y[idxs])
    coefs[b] = ols.coef_
    intercepts[b] = ols.intercept_

boot_df = pd.DataFrame(coefs, columns=FEATURES)
summary = pd.DataFrame({
    'feature':   FEATURES,
    'boot_mean': boot_df.mean().values,
    'boot_std':  boot_df.std().values,
    'boot_cv':   (boot_df.std() / boot_df.mean().abs()).replace([np.inf], np.nan).values,
    'q025':      boot_df.quantile(0.025).values,
    'q975':      boot_df.quantile(0.975).values,
}).round(4)
display(summary)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=boot_df, ax=ax)
ax.axhline(0, color='k', ls='--', alpha=0.5)
ax.set_title(f'Grouped-bootstrap OLS coefficients  ({n_boot} resamples)')
ax.set_ylabel('coefficient')
plt.tight_layout(); plt.show()
print('Interpretation: a feature whose box crosses 0 and has a wide IQR is\n'
      'not reliably identified under OLS, even if the model predicts well.')

## 2. Reparameterization — the Δs target

Write the assumed model as
$$s_t \;=\; \alpha\,s_{t-1} \;+\; \beta_+\,(m_t - v)_+ \;-\; \beta_-\,(v - m_t)_+ \;+\; \mathbf{c}^\top \mathbf{z}_t \;+\; \varepsilon_t$$
where `z_t` collects the auxiliary features. OLS is in trouble because `s_{t-1}` explains a huge fraction of `s_t` by itself.

Subtract `s_{t-1}` from both sides:
$$\underbrace{s_t - s_{t-1}}_{\Delta s_t} \;=\; (\alpha - 1)\,s_{t-1} \;+\; \beta_+\,(m_t - v)_+ \;-\; \beta_-\,(v - m_t)_+ \;+\; \mathbf{c}^\top \mathbf{z}_t \;+\; \varepsilon_t$$

The hinge coefficients `β_+, β_-, c` are **identical** in the two parameterizations. The only thing that changes is that the AR coefficient shifts from `α` (near 1) to `α − 1` (near 0). The target variance is much smaller — it's *concession size* rather than *price level* — so the problem is no longer dominated by the `s_{t-1}`-explains-`s_t` trivial relationship.

We refit and compare the two parameterizations. The hinge coefficients should agree. The AR coefficient on `s_{t-1}` shifts by exactly 1 — that's the check.

In [ ]:
y_delta = sub['delta_s'].values  # = s_t - s_prev

ols_st   = LinearRegression().fit(X, y)
ols_delta = LinearRegression().fit(X, y_delta)

coef_table = pd.DataFrame({
    'feature': ['(intercept)'] + FEATURES,
    'OLS_target_s_t':    [ols_st.intercept_,    *ols_st.coef_],
    'OLS_target_delta_s':[ols_delta.intercept_, *ols_delta.coef_],
}).round(4)
coef_table['diff'] = (coef_table['OLS_target_s_t'] -
                      coef_table['OLS_target_delta_s']).round(4)
display(coef_table)
print("CHECK: 'diff' should be 1.0 for the s_prev row (because α and α−1 differ by 1)\n"
      "and 0 for every other coefficient.  Small deviations are numerical.")

Excellent — the reparameterization is mathematically equivalent. But now the **target** is no longer dominated by a trivially-collinear predictor, and we get a better signal-to-noise for the hinge terms. We move forward using `delta_s` as the target.

We re-run the grouped bootstrap to confirm the hinge coefficients are more stable under this parameterization — they were already well-identified, but the *narrative* of the model is now cleaner.

In [ ]:
rng = np.random.default_rng(0)
coefs_d = np.zeros((n_boot, len(FEATURES)))
for b in range(n_boot):
    sampled = rng.choice(unique_runs, size=len(unique_runs), replace=True)
    idxs = np.concatenate([np.where(groups == r)[0] for r in sampled])
    ols_b = LinearRegression().fit(X[idxs], y_delta[idxs])
    coefs_d[b] = ols_b.coef_

boot_delta = pd.DataFrame(coefs_d, columns=FEATURES)
comp = pd.DataFrame({
    'feature':    FEATURES,
    'std_s_t':    boot_df.std().values,
    'std_delta':  boot_delta.std().values,
    'ratio':      (boot_delta.std() / boot_df.std()).values,
}).round(4)
display(comp)

# Side-by-side boxplot
long_df = pd.concat([
    boot_df.assign(target='s_t'),
    boot_delta.assign(target='Δs')
]).melt(id_vars=['target'], var_name='feature', value_name='coef')
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=long_df, x='feature', y='coef', hue='target', ax=ax)
ax.axhline(0, color='k', ls='--', alpha=0.5)
ax.set_title('Bootstrap coefficient distributions — s_t target vs Δs target')
plt.tight_layout(); plt.show()
print('Note: for every feature EXCEPT s_prev the two distributions are\n'
      'identical (the coefficients agree exactly). For s_prev, the Δs\n'
      'parameterization is centered near α−1 ≈ −0.35 instead of α ≈ 0.65.')

## 3. Robust methods in the Δs parameterization

Now we compare OLS, Ridge (L2 penalty), ElasticNet (mixed L1/L2), PLS (Partial Least Squares), and PCR (Principal-Component Regression). These all handle collinear features, but in different ways:

- **Ridge**: shrinks all coefficients toward 0 (requires feature scaling). Keeps all features.
- **ElasticNet**: Ridge + some Lasso-like zeroing-out. Can be sparse.
- **PLS**: builds `k` latent components that are maximally correlated with `y` *and* orthogonal to each other. Extremely robust but coefficients are interpreted via the latent structure.
- **PCR**: similar to PLS but components come from X alone (ignore y when constructing them).

For all methods we scale features so the penalty acts uniformly. We cross-validate at the **run level** (GroupKFold) to avoid leakage.

In [ ]:
gkf = GroupKFold(n_splits=5)
models = {
    'OLS':        Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
    'Ridge':      Pipeline([('sc', StandardScaler()),
                            ('m', RidgeCV(alphas=np.logspace(-3, 3, 40)))]),
    'ElasticNet': Pipeline([('sc', StandardScaler()),
                            ('m', ElasticNetCV(l1_ratio=[.1,.3,.5,.7,.9],
                                                alphas=np.logspace(-3, 1, 30),
                                                cv=3, max_iter=20000))]),
    'PLS-2':      Pipeline([('sc', StandardScaler()), ('m', PLSRegression(n_components=2))]),
    'PLS-3':      Pipeline([('sc', StandardScaler()), ('m', PLSRegression(n_components=3))]),
    'PCR-3':      Pipeline([('sc', StandardScaler()),
                            ('pca', PCA(n_components=3)),
                            ('m', LinearRegression())]),
}

rows = []
for name, model in models.items():
    scores = cross_val_score(model, X, y_delta, cv=gkf, groups=groups,
                              scoring='r2')
    rmse = -cross_val_score(model, X, y_delta, cv=gkf, groups=groups,
                             scoring='neg_root_mean_squared_error')
    rows.append({'model': name,
                  'cv_r2_mean': scores.mean(),
                  'cv_r2_std':  scores.std(),
                  'cv_rmse_mean': rmse.mean()})
cv_table = pd.DataFrame(rows).round(4)
display(cv_table.sort_values('cv_r2_mean', ascending=False))

### Coefficient comparison — the key robustness test

Scaling-aware trick: for Ridge/ElasticNet the coefficients live in the standardized feature space. To map them back to the interpretable raw-data scale we divide by the feature standard deviations. This way all methods' coefficients are on the same axis.

In [ ]:
def extract_scaled_coefs(pipeline, feature_names):
    sc = pipeline.named_steps['sc']
    model = pipeline.named_steps['m']
    std = sc.scale_
    if 'pca' in pipeline.named_steps:
        # PCR: coefficients live in PCA latent space -> rotate back
        pca = pipeline.named_steps['pca']
        beta_std = pca.components_.T @ np.asarray(model.coef_).flatten()
    else:
        beta_std = np.asarray(model.coef_).flatten()
    # Map back to raw units:  y = β_std · z  and z = (x−μ)/σ  ⇒  β_raw = β_std / σ
    beta_raw = beta_std / std
    return pd.Series(beta_raw, index=feature_names)

fit = {}
for name, model in models.items():
    model.fit(X, y_delta)
    fit[name] = model

coef_compare = pd.DataFrame({
    name: extract_scaled_coefs(m, FEATURES) for name, m in fit.items()
}).round(4)
display(coef_compare)

fig, ax = plt.subplots(figsize=(12, 5))
coef_compare.plot.bar(ax=ax)
ax.axhline(0, color='k', ls='--', alpha=0.5)
ax.set_title('Coefficients across methods — Δs target, raw-scale units')
ax.set_ylabel('β (raw scale)')
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

**Reading the table.**
- If the hinge coefficients `β(diff_pos)` and `β(diff_neg)` agree across OLS, Ridge, and PLS, the hinge form is a *real* signal, not an artefact of which penalty we chose.
- If the `s_prev` coefficient differs wildly across methods — expected, because that's the direction the collinearity lives in — don't panic: the *predictive performance* still agrees. This is exactly what robust methods buy you: the pieces you care about are identified even when the pieces you don't care about (the AR level term) are under-identified.

### Bootstrap stability of the robust methods

In [ ]:
n_boot2 = 150
rng = np.random.default_rng(1)
boot_by_method = {name: np.zeros((n_boot2, len(FEATURES))) for name in models}
for b in range(n_boot2):
    sampled = rng.choice(unique_runs, size=len(unique_runs), replace=True)
    idxs = np.concatenate([np.where(groups == r)[0] for r in sampled])
    Xb = X[idxs]; yb = y_delta[idxs]
    for name, pipe in models.items():
        try:
            pipe.fit(Xb, yb)
            boot_by_method[name][b] = extract_scaled_coefs(pipe, FEATURES).values
        except Exception:
            boot_by_method[name][b] = np.nan

stab_rows = []
for name, M in boot_by_method.items():
    for j, feat in enumerate(FEATURES):
        stab_rows.append({'method': name, 'feature': feat,
                          'mean': np.nanmean(M[:, j]),
                          'std':  np.nanstd(M[:, j])})
stab_df = pd.DataFrame(stab_rows).round(4)
pivot_std = stab_df.pivot(index='feature', columns='method', values='std').round(3)
print('Bootstrap standard deviation of coefficients (lower = more stable):')
display(pivot_std[list(models.keys())])

# Box-plot: compare OLS vs Ridge vs PLS-3
highlight = ['OLS', 'Ridge', 'PLS-3']
fig, axes = plt.subplots(1, len(FEATURES), figsize=(3*len(FEATURES), 4), sharey=False)
for j, feat in enumerate(FEATURES):
    data = [boot_by_method[m][:, j][~np.isnan(boot_by_method[m][:, j])]
            for m in highlight]
    axes[j].boxplot(data, labels=highlight)
    axes[j].axhline(0, color='k', ls='--', alpha=0.5)
    axes[j].set_title(feat)
plt.suptitle('Coefficient bootstrap distributions — robust methods (Δs target)', y=1.02)
plt.tight_layout(); plt.show()

## 4. Final recommended model

Putting it all together:

- **Target:** `Δs = s_t − s_{t-1}`.
- **Features:** `diff_pos`, `diff_neg`, `s_prev`, `m_prev`, `ds_pos`, `ds_neg` (hinge model).
- **Estimator:** Ridge with cross-validated α (via RidgeCV) on standardized features, mapped back to raw units.
- **Validation:** GroupKFold at the `run_idx` level — no within-run leakage.

We report the final coefficients with grouped-bootstrap 95% CIs and the held-out R² on a group-level split.

In [ ]:
# Held-out split: 30% of runs
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
tr_idx, te_idx = next(gss.split(X, y_delta, groups))

final_pipe = Pipeline([('sc', StandardScaler()),
                        ('m', RidgeCV(alphas=np.logspace(-3, 3, 40)))])
final_pipe.fit(X[tr_idx], y_delta[tr_idx])
y_pred = final_pipe.predict(X[te_idx])
final_r2 = r2_score(y_delta[te_idx], y_pred)
final_rmse = np.sqrt(mean_squared_error(y_delta[te_idx], y_pred))
final_mae = mean_absolute_error(y_delta[te_idx], y_pred)
print(f'Held-out run-level split (30% of runs):')
print(f'  R² on Δs = {final_r2:.4f}')
print(f'  RMSE     = {final_rmse:.4f}')
print(f'  MAE      = {final_mae:.4f}')

# Bootstrap CI for final model
rng = np.random.default_rng(2)
tr_groups = np.unique(groups[tr_idx])
n_boot3 = 300
boot_final = np.zeros((n_boot3, len(FEATURES)))
boot_int   = np.zeros(n_boot3)
for b in range(n_boot3):
    sampled = rng.choice(tr_groups, size=len(tr_groups), replace=True)
    idxs = np.concatenate([np.where(groups == r)[0] for r in sampled])
    pipe = Pipeline([('sc', StandardScaler()),
                      ('m', RidgeCV(alphas=np.logspace(-3, 3, 40)))])
    pipe.fit(X[idxs], y_delta[idxs])
    boot_final[b] = extract_scaled_coefs(pipe, FEATURES).values
    boot_int[b]   = pipe.named_steps['m'].intercept_  # in standardized space

ci_df = pd.DataFrame({
    'feature':  FEATURES,
    'point':    extract_scaled_coefs(final_pipe, FEATURES).values,
    'boot_mean': boot_final.mean(0),
    'ci_lo':    np.quantile(boot_final, 0.025, axis=0),
    'ci_hi':    np.quantile(boot_final, 0.975, axis=0),
}).round(4)
display(ci_df)

fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(range(len(FEATURES)), ci_df['point'],
            yerr=[ci_df['point'] - ci_df['ci_lo'],
                  ci_df['ci_hi'] - ci_df['point']],
            fmt='o', capsize=4, lw=2)
ax.axhline(0, color='k', ls='--', alpha=0.5)
ax.set_xticks(range(len(FEATURES)))
ax.set_xticklabels(FEATURES, rotation=20)
ax.set_title('Final Ridge (Δs target) — coefficients with 95% grouped-bootstrap CIs')
ax.set_ylabel('β (raw scale)')
plt.tight_layout(); plt.show()

## 5. The functional form, in words

With the fitted coefficients from above, the seller's decision rule on the Negotiation region has the compact form

$$\boxed{\;
s_t \;=\; s_{t-1} \;+\; \hat\beta_+ (m_t - v)_+ \;-\; \hat\beta_- (v - m_t)_+ \;+\; \hat\gamma_s\, s_{t-1} \;+\; \hat\gamma_m\, m_{t-1} \;+\; \hat\delta_+ \Delta s_{t-1,+} \;-\; \hat\delta_- \Delta s_{t-1,-} \;+\; \hat\alpha_0
\;}$$

with `γ_s ≈ 0` (the collinearity-robust estimate — the AR component is absorbed into the `s_{t-1}` on the left-hand side). Reading the signs:
- `β_+ > 0` means: when the merchant offers something above break-even, the seller *moves toward* them (a concession).
- `β_- < 0` (expressed with a minus sign out front so the coefficient itself is positive): offers below break-even cause the seller to *increase* their counter-quote (push back / hold firm).
- The magnitudes of `β_+` and `β_-` are the two numbers that *define the seller strategy*. Notebook 03 estimates these per strategy and shows they form distinct fingerprints.

### What the collinearity diagnostics told us

1. The raw-`s_t`-target OLS fit has a well-identified prediction but unstable individual coefficients; in particular, `s_{t-1}` absorbs most of the level and its coefficient is near 1 but imprecise.
2. Switching to `Δs_t` as the target eliminates the trivial AR collinearity. The hinge coefficients are unchanged, as the algebra requires.
3. Ridge, PLS, and PCR agree with OLS on the hinge coefficients (robustness check passed) and produce tighter bootstrap intervals because they shrink the near-null directions of the design matrix.

Notebook 03 picks up the final form and fits it separately to each of the six seller strategies.